# Imports and environment setup
This notebook operates the **Mikrotron EoSens Creation 2.0 CX12** camera over CoaXPress (Euresys egrabber).
It provides an interactive AOI selection GUI, safe hardware limit resolution, synchronized audio playback, 
and high-throughput multi-part buffer recording.

In [1]:
import copy
import ctypes as ct
import gc
import json
import math
import os
import re
import sys
import shutil
import time
import warnings
from dataclasses import dataclass
from datetime import datetime
from pathlib import Path
from threading import Thread
from typing import Optional, Tuple

import cv2
import numpy as np
import winsound
from egrabber import *

# ==============================================================================
# --- Audio & Interferometry Reconstruction Imports (Optional) ----------------
# ==============================================================================
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from scipy.signal import butter, filtfilt, resample
import scipy.io.wavfile as wavfile
from sklearn.decomposition import TruncatedSVD
from IPython.display import Audio, display
# ==============================================================================

warnings.filterwarnings("default")

def create_default_opt():
    return {
        "camera_model": "EoSensCreation2.0CX12",
        "interface": "CXP12_X4",
        "cam_params": {}
    }

if "run_opt" not in globals():
    run_opt = create_default_opt()

N_cams = 1

# GUI and recording helpers
Run once. This cell defines:
1. `AOILimits`: Snaps arbitrary dragging rectangles to EoSens hardware grids (16x4 px).
2. `EoSensCamera`: Complete wrapper over EGenTL / EGrabber for CXP-12 cameras.
3. `_PreviewCamera` & `_SelectorWindow`: Full-sensor live preview with pan, zoom, screenshot (Ctrl+S), and brightness control.
4. `configure_recording_camera`: Resolves hardware max FPS, sets multi-part buffers, and checks exposure feasibility.
5. `export_recording_avi`: Exports recorded uint8 frames to an AVI video file.

In [2]:
ROI = Tuple[int, int, int, int]
_SHARED_GENTL: Optional[EGenTL] = None

def get_shared_gentl() -> EGenTL:
    """Maintain a single process-wide GenTL producer instance (singleton on sys)."""
    if hasattr(sys, "_eosens_gentl_singleton") and sys._eosens_gentl_singleton is not None:
        return sys._eosens_gentl_singleton

    try:
        sys._eosens_gentl_singleton = EGenTL()
    except Exception:
        disconnect_camera()
        sys._eosens_gentl_singleton = EGenTL()

    return sys._eosens_gentl_singleton


def disconnect_camera():
    """Safely release all camera, grabber, and thread handles across the notebook."""
    if "imager" in globals():
        im = globals().get("imager")
        if im is not None:
            if hasattr(im, "grabber") and im.grabber is not None:
                try:
                    im.grabber.stop()
                except Exception:
                    pass
                im.grabber = None
            if hasattr(im, "camera"):
                im.camera = None
        globals()["imager"] = None

    for var_name in ["cam", "active_cam", "recording_cam"]:
        if var_name in globals():
            c = globals().get(var_name)
            if c is not None and hasattr(c, "close"):
                try:
                    c.close()
                except Exception:
                    pass
            globals()[var_name] = None

    if "cams" in globals():
        for c in globals().get("cams", []):
            if c is not None and hasattr(c, "close"):
                try:
                    c.close()
                except Exception:
                    pass
        globals()["cams"] = []

    gc.collect()
    time.sleep(0.2)


def stretch_contrast(image: np.ndarray) -> np.ndarray:
    """Normalize image dynamically to 0-255 for visualization."""
    return cv2.normalize(image, None, 0, 255, cv2.NORM_MINMAX, dtype=cv2.CV_8U)


def estimate_eograbber_max_fps(height: int) -> float:
    """Calculate the EoSens Creation 2.0 CX12 maximum frame rate directly from ROI height.
    Row line period = 0.4096 µs, frame readout overhead = 2.5856 µs.
    """
    if height <= 0:
        return 0.0
    frame_time_us = int(height) * 0.4096 + 2.5856
    return 1000000.0 / frame_time_us


@dataclass(frozen=True)
class AOILimits:
    width: int = 1920
    height: int = 1080
    min_width: int = 1         # No minimum limit on width for selection
    min_height: int = 4        # Hardware floor is 4 px
    x_step: int = 1            # Free horizontal positioning
    y_step: int = 4            # 4 px vertical position step
    width_step: int = 1        # Free horizontal sizing
    height_step: int = 4       # 4 px height increments

    def snap(self, first: Tuple[float, float], second: Tuple[float, float]) -> ROI:
        """Normalize dragging direction, snap height to 4px grid, and keep within sensor bounds."""
        def axis(a, b, limit, minimum, position_step, size_step):
            lo, hi = sorted((max(0.0, min(float(a), float(limit))),
                             max(0.0, min(float(b), float(limit)))))
            minimum = math.ceil(minimum / size_step) * size_step
            maximum = (limit // size_step) * size_step
            if maximum < minimum:
                raise ValueError("Sensor cannot accommodate the minimum AOI size.")
            start = (math.floor(lo) // position_step) * position_step
            size = math.ceil(max(hi - start, minimum) / size_step) * size_step
            size = min(size, maximum)
            start = min(start, ((limit - size) // position_step) * position_step)
            return int(start), int(size)

        x, w = axis(first[0], second[0], self.width, self.min_width, self.x_step, self.width_step)
        y, h = axis(first[1], second[1], self.height, self.min_height, self.y_step, self.height_step)
        return x, y, w, h


def compute_hardware_aoi(user_x: int, user_y: int, user_w: int, user_h: int) -> Tuple[ROI, int]:
    """Pads user AOI width to at least 128 px and aligns to 16 px for the EoSens sensor.
    Near sensor edges, padding automatically becomes asymmetric to stay strictly within [0, 1920].
    Returns ((hw_x, hw_y, hw_w, hw_h), crop_x_offset)."""
    hw_h = int(user_h)
    hw_y = int(user_y)

    needed_w = max(128, int(user_w))
    hw_w = int(math.ceil(needed_w / 16.0)) * 16

    center_x = user_x + user_w / 2.0
    hw_x_raw = int(round((center_x - hw_w / 2.0) / 16.0)) * 16

    # Asymmetrically clamp within sensor boundary [0, 1920 - hw_w]
    hw_x = max(0, min(1920 - hw_w, hw_x_raw))

    if user_x < hw_x:
        hw_x = int(math.floor(user_x / 16.0)) * 16
    if user_x + user_w > hw_x + hw_w:
        hw_w = int(math.ceil((user_x + user_w - hw_x) / 16.0)) * 16
        hw_x = max(0, min(1920 - hw_w, hw_x))

    crop_x = int(user_x - hw_x)
    return (int(hw_x), int(hw_y), int(hw_w), int(hw_h)), crop_x


class EoSensCamera:
    """Hardware wrapper managing GenTL and EGrabber state with hardware padding support."""

    def __init__(self, cxp_link: str = "CXP12_X4"):
        self.gentl = get_shared_gentl()
        self.grabber = EGrabber(self.gentl)
        self.grabber.remote.set("CxpLinkConfiguration", cxp_link)
        self.grabber.stream.set("BufferPartCount", 1)
        self.has_gain = False
        try:
            self.grabber.remote.get("Gain")
            self.has_gain = True
        except Exception:
            self.has_gain = False
        self.limits = AOILimits()
        self.user_aoi = (0, 0, 1920, 1080)
        self.hw_aoi = (0, 0, 1920, 1080)
        self.crop_x = 0
        self.update_dimensions()
        self.grabber.realloc_buffers(20)

    def update_dimensions(self):
        self.hw_w = int(self.grabber.remote.get("Width"))
        self.hw_h = int(self.grabber.remote.get("Height"))
        self.w = self.user_aoi[2]
        self.h = self.user_aoi[3]

    def get_aoi(self) -> ROI:
        return self.user_aoi

    def get_hw_aoi(self) -> ROI:
        return self.hw_aoi

    def set_aoi(self, x: int, y: int, w: int, h: int):
        limits = self.limits
        if h < limits.min_height or h % limits.height_step != 0 or h > limits.height:
            raise ValueError(f"Height must be >= {limits.min_height}, multiple of {limits.height_step}, <= {limits.height}")
        if y % limits.y_step != 0 or y + h > limits.height:
            raise ValueError(f"OffsetY must be a multiple of {limits.y_step} and within sensor boundary.")
        if w < 1 or x < 0 or x + w > limits.width:
            raise ValueError("Width and OffsetX must be within the 1920 px sensor boundary.")

        (hw_x, hw_y, hw_w, hw_h), crop_x = compute_hardware_aoi(x, y, w, h)

        self.grabber.remote.set("OffsetX", 0)
        self.grabber.remote.set("OffsetY", 0)
        self.grabber.remote.set("Width", int(hw_w))
        self.grabber.remote.set("Height", int(hw_h))
        self.grabber.remote.set("OffsetX", int(hw_x))
        self.grabber.remote.set("OffsetY", int(hw_y))

        self.user_aoi = (int(x), int(y), int(w), int(h))
        self.hw_aoi = (int(hw_x), int(hw_y), int(hw_w), int(hw_h))
        self.crop_x = crop_x
        self.update_dimensions()

    def reset_aoi(self):
        self.grabber.remote.set("OffsetX", 0)
        self.grabber.remote.set("OffsetY", 0)
        self.grabber.remote.set("Width", 1920)
        self.grabber.remote.set("Height", 1080)
        self.user_aoi = (0, 0, 1920, 1080)
        self.hw_aoi = (0, 0, 1920, 1080)
        self.crop_x = 0
        self.update_dimensions()

    def get_exposure_us(self) -> float:
        return float(self.grabber.remote.get("ExposureTime"))

    def set_exposure_us(self, val_us: float):
        val_us = max(2.0, float(val_us))
        curr_fps = self.get_frame_rate()
        if curr_fps > 0:
            max_exp = max(2.0, (1000000.0 / curr_fps) - 2.5)
            val_us = min(val_us, max_exp)
        self.grabber.remote.set("ExposureTime", float(val_us))

    def get_frame_rate(self) -> float:
        return float(self.grabber.remote.get("AcquisitionFrameRate"))

    def set_frame_rate(self, fps: float):
        max_fps = self.get_max_frame_rate()
        target = min(float(fps), max_fps)
        self.grabber.remote.set("AcquisitionFrameRate", float(target))

    def get_max_frame_rate(self) -> float:
        return float(self.grabber.remote.get("AcquisitionFrameRateMax"))

    def get_gain(self) -> float:
        if self.has_gain:
            try:
                return float(self.grabber.remote.get("Gain"))
            except Exception:
                return 3.0
        return 3.0

    def set_gain(self, gain_val: float):
        if self.has_gain:
            val_discrete = float(max(1, min(3, int(round(float(gain_val))))))
            self.grabber.remote.set("Gain", val_discrete)

    def set_buffer_part_count(self, count: int):
        self.grabber.stream.set("BufferPartCount", int(count))
        self.grabber.realloc_buffers(20)

    def close(self):
        if hasattr(self, "grabber") and self.grabber is not None:
            try:
                self.grabber.stop()
            except Exception:
                pass
            try:
                self.grabber.flush_buffers()
            except Exception:
                pass
            del self.grabber
            self.grabber = None


def _build_gamma_lut(gamma: float) -> Optional[np.ndarray]:
    gamma = max(0.01, min(2.0, float(gamma)))
    if abs(gamma - 1.0) < 1e-3:
        return None
    inv_gamma = 1.0 / gamma
    table = np.array([((i / 255.0) ** inv_gamma) * 255.0 for i in range(256)])
    return np.clip(table, 0, 255).astype(np.uint8)


class _PreviewCamera:
    def __init__(self, camera: EoSensCamera):
        self.camera = camera
        self.grabber = camera.grabber
        self.capture_active = False
        self.saved = None
        self.last_frame_time = 0.0
        self.gamma = 1.0
        self.gamma_lut = None

    def start(self, exposure_us: float, preview_fps: float):
        self.saved = {
            "user_aoi": self.camera.user_aoi,
            "hw_aoi": self.camera.hw_aoi,
            "crop_x": self.camera.crop_x,
            "fps": self.camera.get_frame_rate(),
            "exposure_us": self.camera.get_exposure_us(),
            "buffer_part_count": int(self.grabber.stream.get("BufferPartCount")),
            "gain": self.camera.get_gain() if self.camera.has_gain else 3.0,
        }

        try:
            self.grabber.stop()
        except Exception:
            pass

        self.camera.set_buffer_part_count(1)
        self.camera.reset_aoi()
        self.limits = self.camera.limits
        self.width = 1920
        self.height = 1080

        max_preview_fps = self.camera.get_max_frame_rate()
        target_fps = min(float(preview_fps), max_preview_fps)
        self.camera.set_frame_rate(target_fps)
        self.fps = self.camera.get_frame_rate()

        self.exposure_min_us = 2.0
        self.exposure_max_us = max(2.1, (1000000.0 / self.fps) - 5.0)
        self.exposure_us = max(self.exposure_min_us, min(self.exposure_max_us, float(exposure_us)))
        self.camera.set_exposure_us(self.exposure_us)

        self.has_gain = self.camera.has_gain
        self.gain = 3.0
        if self.has_gain:
            self.camera.set_gain(self.gain)

        self.set_gamma(1.0)

        self.grabber.realloc_buffers(3)
        self.grabber.flush_buffers()
        self.grabber.start()
        self.capture_active = True
        self.last_frame_time = time.monotonic()

    def set_exposure(self, us_val: float) -> float:
        us_val = max(self.exposure_min_us, min(self.exposure_max_us, float(us_val)))
        self.camera.set_exposure_us(us_val)
        self.exposure_us = us_val
        return self.exposure_us

    def set_gain(self, val: float) -> float:
        if self.has_gain:
            val_discrete = float(max(1, min(3, int(round(float(val))))))
            self.camera.set_gain(val_discrete)
            self.gain = val_discrete
        return self.gain

    def set_gamma(self, val: float) -> float:
        self.gamma = max(0.0, min(2.0, float(val)))
        self.gamma_lut = _build_gamma_lut(self.gamma)
        return self.gamma

    def latest_frame(self) -> Optional[np.ndarray]:
        if not self.capture_active:
            return None

        frame = None
        w, h = self.width, self.height
        buf_size = w * h

        while True:
            try:
                with Buffer(self.grabber, timeout=0) as buffer:
                    ptr = buffer.get_info(BUFFER_INFO_BASE, INFO_DATATYPE_PTR)
                    data = ct.cast(ptr, ct.POINTER(ct.c_ubyte * buf_size)).contents
                    frame = np.frombuffer(data, count=buf_size, dtype=np.uint8).reshape((h, w)).copy()
                    self.last_frame_time = time.monotonic()
            except Exception:
                break

        if frame is None:
            try:
                with Buffer(self.grabber, timeout=30) as buffer:
                    ptr = buffer.get_info(BUFFER_INFO_BASE, INFO_DATATYPE_PTR)
                    data = ct.cast(ptr, ct.POINTER(ct.c_ubyte * buf_size)).contents
                    frame = np.frombuffer(data, count=buf_size, dtype=np.uint8).reshape((h, w)).copy()
                    self.last_frame_time = time.monotonic()
            except Exception:
                pass

        if frame is not None and self.gamma_lut is not None:
            frame = cv2.LUT(frame, self.gamma_lut)

        return frame

    def close(self):
        if self.capture_active:
            try:
                self.grabber.stop()
            except Exception:
                pass
            self.capture_active = False

        try:
            self.grabber.flush_buffers()
        except Exception:
            pass

        if self.saved is not None:
            old = self.saved
            try:
                self.camera.set_buffer_part_count(old["buffer_part_count"])
            except Exception:
                pass
            try:
                self.camera.set_aoi(*old["user_aoi"])
            except Exception:
                pass
            try:
                max_fps = self.camera.get_max_frame_rate()
                target_fps = min(float(old["fps"]), max_fps)
                self.camera.set_frame_rate(target_fps)
            except Exception:
                pass
            try:
                max_exp = max(2.0, (1000000.0 / self.camera.get_frame_rate()) - 2.5)
                target_exp = min(float(old["exposure_us"]), max_exp)
                self.camera.set_exposure_us(target_exp)
            except Exception:
                pass
            if self.has_gain:
                try:
                    self.camera.set_gain(old.get("gain", 3.0))
                except Exception:
                    pass
            self.saved = None


class _SelectorWindow:
    def __init__(self, root, camera: _PreviewCamera, bayer_code=None):
        import tkinter as tk
        from tkinter import ttk
        from PIL import Image, ImageDraw, ImageTk

        self.tk, self.ttk = tk, ttk
        self.Image, self.ImageDraw, self.ImageTk = Image, ImageDraw, ImageTk
        self.root, self.camera = root, camera
        self.bayer_code = bayer_code
        self.roi = self.result = self.anchor = None
        self.error = None
        self.last_image = self.photo = None
        self.transform = None
        self.timer = None
        self.pending_exposure = None
        self.pending_gain = None
        self.pending_gamma = None
        self.closed = False
        self.updating_controls = True

        self.view_zoom = 1.0
        self.view_center = [camera.width / 2.0, camera.height / 2.0]
        self.max_view_zoom = 32.0
        self.pan_anchor = None
        self.pan_start_center = None

        root.title("EoSens Setup | Preview Brightness + AOI Selection")
        w_win = min(1400, max(720, root.winfo_screenwidth() - 100))
        h_win = min(900, max(520, root.winfo_screenheight() - 100))
        root.geometry(f"{w_win}x{h_win}")
        root.minsize(800, 640)

        style = ttk.Style(root)
        if "clam" in style.theme_names():
            style.theme_use("clam")
        style.configure("Title.TLabel", font=("Arial", 15, "bold"))
        style.configure("Value.TLabel", font=("Consolas", 11, "bold"))

        root.columnconfigure(0, weight=1)
        root.rowconfigure(1, weight=1)

        ttk.Label(root, text="Select recording area (Width free; Height in 4 px steps)",
                  style="Title.TLabel", padding=(16, 12)).grid(row=0, column=0, columnspan=2, sticky="w")

        self.canvas = tk.Canvas(root, background="#151b24", highlightthickness=0, cursor="crosshair")
        self.canvas.grid(row=1, column=0, sticky="nsew", padx=(12, 6), pady=(0, 12))

        panel = ttk.Frame(root, padding=(14, 0, 18, 12), width=330)
        panel.grid(row=1, column=1, sticky="ns")
        panel.columnconfigure(0, weight=1)

        ttk.Label(panel, text="PREVIEW ONLY", font=("Arial", 11, "bold")).grid(row=0, column=0, sticky="w", pady=(0, 4))
        ttk.Label(panel, text="Live brightness is temporary.\nRecording settings are configured next.",
                  wraplength=280).grid(row=1, column=0, sticky="w", pady=(0, 12))

        # Exposure controls
        self.exposure_text = tk.StringVar(root, value=f"{camera.exposure_us:.0f}")
        self.exposure_log = tk.DoubleVar(root, value=math.log10(max(camera.exposure_min_us, camera.exposure_us)))
        ttk.Label(panel, text="Exposure (µs):").grid(row=2, column=0, sticky="w")
        entry = ttk.Entry(panel, textvariable=self.exposure_text)
        entry.grid(row=3, column=0, sticky="ew", pady=4)
        entry.bind("<Return>", self._exposure_entry)
        entry.bind("<KP_Enter>", self._exposure_entry)
        entry.bind("<FocusOut>", self._exposure_entry)

        high = max(camera.exposure_max_us, camera.exposure_min_us * 1.0001)
        ttk.Scale(panel, from_=math.log10(camera.exposure_min_us), to=math.log10(high),
                  variable=self.exposure_log, command=self._exposure_slider).grid(row=4, column=0, sticky="ew")
        ttk.Label(panel, text=f"Range: {camera.exposure_min_us:.0f} - {camera.exposure_max_us:.0f} µs",
                  wraplength=280).grid(row=5, column=0, sticky="w", pady=(2, 10))

        # Gain controls
        self.gain_text = tk.StringVar(root, value=f"Gain: {int(round(camera.gain))}x")
        self.gain_value = tk.DoubleVar(root, value=int(round(camera.gain)))
        ttk.Label(panel, textvariable=self.gain_text).grid(row=6, column=0, sticky="w")
        gain_scale = ttk.Scale(panel, from_=1, to=3, variable=self.gain_value, command=self._gain_slider)
        gain_scale.grid(row=7, column=0, sticky="ew", pady=(4, 10))

        # Gamma controls
        self.gamma_text = tk.StringVar(root, value=f"Gamma: {camera.gamma:.2f}")
        self.gamma_value = tk.DoubleVar(root, value=camera.gamma)
        ttk.Label(panel, textvariable=self.gamma_text).grid(row=8, column=0, sticky="w")
        gamma_scale = ttk.Scale(panel, from_=0.0, to=2.0, variable=self.gamma_value, command=self._gamma_slider)
        gamma_scale.grid(row=9, column=0, sticky="ew", pady=(4, 2))
        ttk.Label(panel, text="Range: 0.00 - 2.00", wraplength=280).grid(row=10, column=0, sticky="w", pady=(0, 10))

        ttk.Separator(panel).grid(row=11, column=0, sticky="ew", pady=6)

        # AOI Display with Real-time MAX_FPS Calculation
        ttk.Label(panel, text="RECORDING AOI", font=("Arial", 11, "bold")).grid(row=12, column=0, sticky="w", pady=(6, 6))
        self.roi_text = tk.StringVar(root, value="AOI_X       = -\nAOI_Y       = -\nAOI_WIDTH   = -\nAOI_HEIGHT  = -\nMAX_FPS     = -")
        ttk.Label(panel, textvariable=self.roi_text, style="Value.TLabel", justify="left").grid(row=13, column=0, sticky="w")

        ttk.Label(panel, text="Max FPS is computed from height.\nWidth < 128 px padded & auto-cropped.",
                  font=("Arial", 9, "italic"), foreground="#4a8f9f", wraplength=280).grid(row=14, column=0, sticky="w", pady=(4, 8))

        ttk.Separator(panel).grid(row=15, column=0, sticky="ew", pady=6)

        # Viewport controls
        ttk.Label(panel, text="VIEW", font=("Arial", 11, "bold")).grid(row=16, column=0, sticky="w", pady=(4, 2))
        self.zoom_text = tk.StringVar(root, value="Zoom: 1.00x")
        ttk.Label(panel, textvariable=self.zoom_text).grid(row=17, column=0, sticky="w")
        ttk.Label(panel, text="Wheel: zoom at cursor\nMiddle-button drag: pan", wraplength=280).grid(row=18, column=0, sticky="w", pady=(2, 6))

        ttk.Button(panel, text="Reset all", command=self.reset_view).grid(row=19, column=0, sticky="ew", pady=(0, 4))
        ttk.Button(panel, text="Capture screenshot  [Ctrl+S]", command=self.capture_screenshot).grid(row=20, column=0, sticky="ew", pady=(0, 6))

        # Actions
        ttk.Button(panel, text="Confirm AOI  [Enter]", command=self.confirm).grid(row=21, column=0, sticky="ew", pady=4)
        ttk.Button(panel, text="Clear selection", command=self.clear).grid(row=22, column=0, sticky="ew", pady=4)
        ttk.Button(panel, text="Cancel  [Esc]", command=self.cancel).grid(row=23, column=0, sticky="ew", pady=4)

        self.status = tk.StringVar(root, value="Left-drag selects AOI. Wheel zooms; middle-drag pans. Enter confirms.")
        ttk.Label(root, textvariable=self.status, padding=(14, 4, 14, 10), wraplength=1100).grid(row=2, column=0, columnspan=2, sticky="w")

        # Bindings
        self.canvas.bind("<ButtonPress-1>", self._press)
        self.canvas.bind("<B1-Motion>", self._drag)
        self.canvas.bind("<ButtonRelease-1>", self._release)
        self.canvas.bind("<MouseWheel>", self._wheel)
        self.canvas.bind("<Button-4>", self._wheel)
        self.canvas.bind("<Button-5>", self._wheel)
        self.canvas.bind("<ButtonPress-2>", self._pan_press)
        self.canvas.bind("<B2-Motion>", self._pan_drag)
        self.canvas.bind("<ButtonRelease-2>", self._pan_release)
        self.canvas.bind("<Configure>", lambda event: self._render())

        root.bind("<Return>", lambda event: self.confirm())
        root.bind("<KP_Enter>", lambda event: self.confirm())
        root.bind("<Escape>", lambda event: self.cancel())
        root.bind("<Control-s>", lambda event: self.capture_screenshot())
        root.bind("<Control-S>", lambda event: self.capture_screenshot())
        root.protocol("WM_DELETE_WINDOW", self.cancel)

        self.canvas.focus_set()
        self.updating_controls = False
        self.timer = root.after(0, self._tick)

    def _exposure_entry(self, event=None):
        if self.closed:
            return "break"
        try:
            val = float(self.exposure_text.get())
            if not math.isfinite(val) or val <= 0:
                raise ValueError
            val = max(self.camera.exposure_min_us, min(self.camera.exposure_max_us, val))
            self.pending_exposure = val
            self.updating_controls = True
            try:
                self.exposure_log.set(math.log10(val))
            finally:
                self.updating_controls = False
        except ValueError:
            self.status.set(f"Enter an exposure between {self.camera.exposure_min_us:.0f} and {self.camera.exposure_max_us:.0f} µs.")
        return "break"

    def _exposure_slider(self, val):
        if not self.updating_controls:
            req_us = 10.0 ** float(val)
            req_us = max(self.camera.exposure_min_us, min(self.camera.exposure_max_us, req_us))
            self.pending_exposure = req_us
            self.exposure_text.set(f"{req_us:.0f}")

    def _gain_slider(self, val):
        if not self.updating_controls:
            val_int = int(round(float(val)))
            val_int = max(1, min(3, val_int))
            self.pending_gain = float(val_int)
            self.gain_text.set(f"Gain: {val_int}x")
            self.updating_controls = True
            try:
                self.gain_value.set(val_int)
            finally:
                self.updating_controls = False

    def _gamma_slider(self, val):
        if not self.updating_controls:
            val_float = max(0.0, min(2.0, float(val)))
            self.pending_gamma = val_float
            self.gamma_text.set(f"Gamma: {val_float:.2f}")

    def _tick(self):
        self.timer = None
        if self.closed:
            return
        try:
            if self.pending_exposure is not None:
                val, self.pending_exposure = self.pending_exposure, None
                self.camera.set_exposure(val)

            if self.pending_gain is not None:
                val, self.pending_gain = self.pending_gain, None
                self.camera.set_gain(val)

            if self.pending_gamma is not None:
                val, self.pending_gamma = self.pending_gamma, None
                self.camera.set_gamma(val)

            frame = self.camera.latest_frame()
            if frame is not None:
                if self.bayer_code is not None:
                    frame = cv2.cvtColor(frame, self.bayer_code)
                self.last_image = self.Image.fromarray(frame)
                self._render()
            elif time.monotonic() - self.camera.last_frame_time > 8.0:
                raise RuntimeError("No preview frames for 8 seconds. Check CoaXPress connection.")
        except Exception as exc:
            self.error = exc
            self._finish()
            return

        self.timer = self.root.after(20, self._tick)

    def _clamp_view_center(self, cw, ch, scale):
        w, h = self.camera.width, self.camera.height
        cx, cy = self.view_center
        if w * scale <= cw:
            cx = w / 2.0
        else:
            half = cw / (2.0 * scale)
            cx = max(half, min(w - half, cx))
        if h * scale <= ch:
            cy = h / 2.0
        else:
            half = ch / (2.0 * scale)
            cy = max(half, min(h - half, cy))
        self.view_center[:] = [cx, cy]

    def _render(self):
        if self.closed or self.last_image is None:
            return
        cw, ch = self.canvas.winfo_width(), self.canvas.winfo_height()
        if cw < 2 or ch < 2:
            return

        w, h = self.camera.width, self.camera.height
        fit_scale = min(cw / w, ch / h)
        scale = fit_scale * self.view_zoom
        self._clamp_view_center(cw, ch, scale)
        cx, cy = self.view_center

        ox = cw / 2.0 - cx * scale
        oy = ch / 2.0 - cy * scale
        self.transform = (ox, oy, scale, scale)

        x0 = max(0, int(math.floor((0.0 - ox) / scale)))
        y0 = max(0, int(math.floor((0.0 - oy) / scale)))
        x1 = min(w, int(math.ceil((cw - ox) / scale)))
        y1 = min(h, int(math.ceil((ch - oy) / scale)))
        if x1 <= x0 or y1 <= y0:
            return

        crop = self.last_image.crop((x0, y0, x1, y1))
        dw = max(1, round((x1 - x0) * scale))
        dh = max(1, round((y1 - y0) * scale))

        resampling = getattr(self.Image, "Resampling", self.Image)
        nearest_mode = getattr(resampling, "NEAREST", 0)
        image = crop.resize((dw, dh), nearest_mode)

        self.photo = self.ImageTk.PhotoImage(image, master=self.root)

        ix = ox + x0 * scale
        iy = oy + y0 * scale
        self.canvas.delete("frame")
        self.canvas.create_image(ix, iy, image=self.photo, anchor="nw", tags="frame")
        self.canvas.tag_lower("frame")
        self._overlay()

    def _point(self, event, require_inside=False):
        if self.transform is None:
            return None
        ox, oy, sx, sy = self.transform
        x, y = (event.x - ox) / sx, (event.y - oy) / sy
        if require_inside and not (0 <= x <= self.camera.width and 0 <= y <= self.camera.height):
            return None
        return (max(0.0, min(x, self.camera.width)), max(0.0, min(y, self.camera.height)))

    def _wheel(self, event):
        if self.transform is None:
            return "break"
        point = self._point(event, require_inside=True)
        if point is None:
            return "break"

        if getattr(event, "num", None) == 4:
            steps = 1.0
        elif getattr(event, "num", None) == 5:
            steps = -1.0
        else:
            delta = float(getattr(event, "delta", 0.0))
            if delta == 0:
                return "break"
            steps = 1.0 if delta > 0 else -1.0 if abs(delta) < 120.0 else max(-4.0, min(4.0, delta / 120.0))

        old_zoom = self.view_zoom
        new_zoom = max(1.0, min(self.max_view_zoom, old_zoom * (1.22 ** steps)))
        if abs(new_zoom - old_zoom) < 1e-12:
            return "break"

        cw, ch = self.canvas.winfo_width(), self.canvas.winfo_height()
        fit_scale = min(cw / self.camera.width, ch / self.camera.height)
        new_scale = fit_scale * new_zoom

        self.view_center[0] = point[0] - (event.x - cw / 2.0) / new_scale
        self.view_center[1] = point[1] - (event.y - ch / 2.0) / new_scale
        self.view_zoom = new_zoom
        self.zoom_text.set(f"Zoom: {self.view_zoom:.2f}x")
        self._render()
        return "break"

    def _pan_press(self, event):
        if self.transform is None:
            return "break"
        self.canvas.focus_set()
        self.pan_anchor = (event.x, event.y)
        self.pan_start_center = tuple(self.view_center)
        self.canvas.configure(cursor="fleur")
        return "break"

    def _pan_drag(self, event):
        if self.pan_anchor is None or self.pan_start_center is None or self.transform is None:
            return "break"
        _, _, sx, sy = self.transform
        dx = event.x - self.pan_anchor[0]
        dy = event.y - self.pan_anchor[1]
        self.view_center[0] = self.pan_start_center[0] - dx / sx
        self.view_center[1] = self.pan_start_center[1] - dy / sy
        self._render()
        return "break"

    def _pan_release(self, event):
        if self.pan_anchor is not None:
            self._pan_drag(event)
        self.pan_anchor = None
        self.pan_start_center = None
        self.canvas.configure(cursor="crosshair")
        return "break"

    def reset_view(self):
        self.updating_controls = True
        try:
            self.view_zoom = 1.0
            self.view_center[:] = [self.camera.width / 2.0, self.camera.height / 2.0]
            self.zoom_text.set("Zoom: 1.00x")

            default_exp = 20.0
            default_exp = max(self.camera.exposure_min_us, min(self.camera.exposure_max_us, default_exp))
            self.camera.set_exposure(default_exp)
            self.pending_exposure = None
            self.exposure_text.set(f"{default_exp:.0f}")
            self.exposure_log.set(math.log10(default_exp))

            default_gain = 3.0
            self.camera.set_gain(default_gain)
            self.pending_gain = None
            self.gain_value.set(int(default_gain))
            self.gain_text.set(f"Gain: {int(default_gain)}x")

            default_gamma = 1.0
            self.camera.set_gamma(default_gamma)
            self.pending_gamma = None
            self.gamma_value.set(default_gamma)
            self.gamma_text.set(f"Gamma: {default_gamma:.2f}")

            self.clear()
        finally:
            self.updating_controls = False

        self._render()
        self.status.set("Reset all: view, exposure (20 µs), gain (3x), gamma (1.00), and AOI cleared.")
        self.canvas.focus_set()

    def capture_screenshot(self):
        if self.last_image is None:
            self.status.set("No preview frame available to capture.")
            self.root.bell()
            return
        cw, ch = self.canvas.winfo_width(), self.canvas.winfo_height()
        w, h = self.camera.width, self.camera.height
        if cw < 2 or ch < 2 or self.transform is None:
            x0, y0, x1, y1 = 0, 0, w, h
        else:
            ox, oy, scale, _ = self.transform
            x0 = max(0, int(math.floor((0.0 - ox) / scale)))
            y0 = max(0, int(math.floor((0.0 - oy) / scale)))
            x1 = min(w, int(math.ceil((cw - ox) / scale)))
            y1 = min(h, int(math.ceil((ch - oy) / scale)))
        screenshot = self.last_image.crop((x0, y0, x1, y1))
        if self.roi is not None:
            if screenshot.mode != "RGB":
                screenshot = screenshot.convert("RGB")
            draw = self.ImageDraw.Draw(screenshot)
            rx, ry, rw, rh = self.roi
            draw.rectangle([rx - x0, ry - y0, rx + rw - x0, ry + rh - y0], outline="#37e5ae", width=2)
        try:
            folder = Path("screenshots")
            folder.mkdir(parents=True, exist_ok=True)
            ts = time.strftime("%Y%m%d_%H%M%S")
            fp = folder / f"screenshot_{ts}.png"
            screenshot.save(fp)
            self.status.set(f"Saved: {fp.resolve()}")
        except Exception as exc:
            self.status.set(f"Screenshot save failed: {exc}")
            self.root.bell()

    def _press(self, event):
        self.canvas.focus_set()
        self.anchor = self._point(event, require_inside=True)
        if self.anchor is not None:
            self.roi = self.camera.limits.snap(self.anchor, self.anchor)
            self._overlay()

    def _drag(self, event):
        pt = self._point(event)
        if self.anchor is not None and pt is not None:
            self.roi = self.camera.limits.snap(self.anchor, pt)
            self._overlay()

    def _release(self, event):
        self._drag(event)
        self.anchor = None

    def _overlay(self):
        self.canvas.delete("roi")
        if self.roi is None or self.transform is None:
            return
        x, y, w, h = self.roi
        (hw_x, hw_y, hw_w, hw_h), _ = compute_hardware_aoi(x, y, w, h)
        max_fps = estimate_eograbber_max_fps(h)
        
        self.roi_text.set(
            f"AOI_X       = {x}\n"
            f"AOI_Y       = {y}\n"
            f"AOI_WIDTH   = {w}\n"
            f"AOI_HEIGHT  = {h}\n"
            f"MAX_FPS     = {max_fps:,.0f} fps\n"
            f"(HW Stream: {hw_w}x{hw_h})"
        )
        ox, oy, sx, sy = self.transform
        x1, y1, x2, y2 = ox + x * sx, oy + y * sy, ox + (x + w) * sx, oy + (y + h) * sy
        self.canvas.create_rectangle(x1, y1, x2, y2, outline="#37e5ae", width=2, tags="roi")
        label = f"W={w} H={h} | Max: {max_fps:,.0f} FPS (HW: {hw_w}x{hw_h})"
        item = self.canvas.create_text(12, 12, text=label, fill="white",
                                       font=("Consolas", 11, "bold"), anchor="nw", tags="roi")
        box = self.canvas.bbox(item)
        bg = self.canvas.create_rectangle(box[0] - 6, box[1] - 4, box[2] + 6, box[3] + 4,
                                          fill="#151b24", outline="", tags="roi")
        self.canvas.tag_lower(bg, item)

    def clear(self):
        self.roi = self.anchor = None
        self.canvas.delete("roi")
        self.roi_text.set("AOI_X       = -\nAOI_Y       = -\nAOI_WIDTH   = -\nAOI_HEIGHT  = -\nMAX_FPS     = -")
        self.status.set("Selection cleared. Drag a new rectangle.")
        self.canvas.focus_set()

    def confirm(self):
        if self.roi is None:
            self.status.set("Draw an ROI before confirming.")
            self.root.bell()
            return
        self.result = self.roi
        self._finish()

    def cancel(self):
        self.result = None
        self._finish()

    def _finish(self):
        self.closed = True
        if self.timer is not None:
            self.root.after_cancel(self.timer)
            self.timer = None
        self.root.quit()


def select_aoi_gui(camera: EoSensCamera, *, preview_exposure_us=20.0, preview_fps=60.0,
                   bayer_to_rgb_code=None) -> Optional[ROI]:
    """Launch full-sensor interactive preview window; returns (x, y, w, h) or None."""
    import tkinter as tk
    root = tk.Tk()
    root.withdraw()
    preview_cam = None
    try:
        preview_cam = _PreviewCamera(camera)
        preview_cam.start(float(preview_exposure_us), float(preview_fps))
        gui = _SelectorWindow(root, preview_cam, bayer_code=bayer_to_rgb_code)
        
        root.deiconify()
        root.lift()
        root.attributes("-topmost", True)
        root.focus_force()
        root.after(250, lambda: root.attributes("-topmost", False))
        
        root.mainloop()
        if gui.error is not None:
            raise gui.error
        return gui.result
    finally:
        try:
            root.destroy()
        finally:
            if preview_cam is not None:
                preview_cam.close()


def get_recording_state(camera: EoSensCamera) -> dict:
    aoi = list(camera.get_aoi())
    hw_aoi = list(camera.get_hw_aoi())
    fps = camera.get_frame_rate()
    max_fps = camera.get_max_frame_rate()
    exp_us = camera.get_exposure_us()
    buffer_parts = int(camera.grabber.stream.get("BufferPartCount"))
    gain = int(round(camera.get_gain()))
    return {
        "aoi": aoi,
        "hw_aoi": hw_aoi,
        "crop_x": camera.crop_x,
        "frame_rate_fps": fps,
        "max_frame_rate_fps": max_fps,
        "exposure_us": exp_us,
        "exposure_ms": exp_us / 1000.0,
        "gain": gain,
        "buffer_part_count": buffer_parts,
    }


def configure_recording_camera(camera: EoSensCamera, frame_rate: Optional[float] = None,
                               exposure_us: Optional[float] = None,
                               exposure_ms: Optional[float] = None,
                               gain: float = 3.0,
                               buffer_part_count: int = 100) -> dict:
    try:
        camera.grabber.stop()
    except Exception:
        pass

    camera.grabber.remote.set("AcquisitionMode", "Continuous")

    if exposure_us is not None:
        target_exp_us = float(exposure_us)
    elif exposure_ms is not None:
        target_exp_us = float(exposure_ms) * 1000.0
    else:
        target_exp_us = 20.0
    target_exp_us = max(2.0, target_exp_us)

    hw_max_fps = camera.get_max_frame_rate()
    sensor_overhead_us = 2.5
    max_fps_for_exp = 1000000.0 / (target_exp_us + sensor_overhead_us)

    if frame_rate is None:
        target_fps = min(hw_max_fps, max_fps_for_exp)
    else:
        target_fps = float(frame_rate)
        if target_fps > hw_max_fps:
            raise ValueError(f"Requested {target_fps:.2f} FPS exceeds hardware limit of {hw_max_fps:.2f} FPS for this AOI.")
        if target_fps > max_fps_for_exp:
            raise ValueError(
                f"Requested {target_fps:.2f} FPS cannot fit exposure of {target_exp_us:.1f} µs.\n"
                f"-> Set FPS <= {max_fps_for_exp:.1f} or exposure <= {(1000000.0/target_fps - sensor_overhead_us):.1f} µs."
            )

    camera.set_frame_rate(target_fps)
    camera.set_exposure_us(target_exp_us)
    if camera.has_gain:
        camera.set_gain(max(1.0, min(3.0, float(gain))))

    camera.set_buffer_part_count(buffer_part_count)

    state = get_recording_state(camera)
    state.update({
        "automatic_max_fps": frame_rate is None,
        "requested_frame_rate_fps": target_fps,
        "requested_exposure_us": target_exp_us,
        "requested_exposure_ms": target_exp_us / 1000.0,
        "requested_gain": int(round(gain)),
    })
    return state


def export_recording_avi(path: str | Path, frames: np.ndarray, fps: float,
                         is_color: bool = False, codec: str = "MJPG", bayer_code=None):
    frames = np.asarray(frames)
    fps = float(fps)
    if frames.ndim != 3 or frames.dtype != np.uint8 or len(frames) == 0:
        raise ValueError("AVI export expects non-empty uint8 (N, H, W) array.")

    n, h, w = frames.shape
    pad_h = h % 2
    pad_w = w % 2
    if pad_h > 0 or pad_w > 0:
        frames = np.pad(frames, ((0, 0), (0, pad_h), (0, pad_w)), mode="edge")
        h, w = frames.shape[1:]

    writer = cv2.VideoWriter(str(path), cv2.VideoWriter_fourcc(*codec), fps, (w, h), True)
    if not writer.isOpened():
        raise RuntimeError("OpenCV VideoWriter failed to open. Check codec availability.")
    bayer_conv = cv2.COLOR_BayerGRBG2BGR if bayer_code is None else bayer_code
    try:
        for frame in frames:
            conv = cv2.cvtColor(frame, bayer_conv) if is_color else cv2.cvtColor(frame, cv2.COLOR_GRAY2BGR)
            writer.write(conv)
    finally:
        writer.release()

# Connect camera and choose AOI
Launches the interactive selector window:
- Left-drag to define the recording area (snapped to 16x4 px sensor steps).
- Mouse wheel zooms in/out at the cursor; middle-drag pans.
- Press `Ctrl+S` anytime to save an uncompressed screenshot of the viewport to `screenshots/`.
- Press `Enter` to confirm the selected AOI, or `Esc` to cancel.

In [3]:
# Prevent reconnecting if an acquisition thread is active
if "imager" in globals() and globals().get("imager") is not None and globals()["imager"].is_alive():
    raise RuntimeError("A capture thread is still active. Stop it before re-initializing.")

_recording_settings_valid = False

# Safely disconnect any active camera/grabber handles
disconnect_camera()

print("Initializing EoSens Creation 2.0 CX12 via EGrabber...")
cam = EoSensCamera(cxp_link="CXP12_X4")

try:
    selected_aoi = select_aoi_gui(
        cam,
        preview_exposure_us=20.0,
        preview_fps=60.0,
        bayer_to_rgb_code=None
    )
    if selected_aoi is None:
        raise RuntimeError("AOI selection was cancelled. No new AOI committed.")

    cam.set_aoi(*selected_aoi)

    accepted_aoi = cam.get_aoi()
    if accepted_aoi != selected_aoi:
        raise RuntimeError(f"Camera adjusted requested AOI {selected_aoi} to {accepted_aoi}.")

except BaseException:
    try:
        cam.close()
    except Exception:
        pass
    raise

USE_AOI = True
AOI_X, AOI_Y, AOI_WIDTH, AOI_HEIGHT = accepted_aoi
cams = [cam]
camera_names = ["cam_0"]
N_cams = len(cams)

hw_box = cam.get_hw_aoi()
print(f"Successfully initialized EoSens camera.")
print(f"Selected AOI:  (x={AOI_X}, y={AOI_Y}, width={AOI_WIDTH}, height={AOI_HEIGHT})")
print(f"Hardware AOI:  (x={hw_box[0]}, y={hw_box[1]}, width={hw_box[2]}, height={hw_box[3]}) [crop_x = {cam.crop_x}]")
print("Next: configure Recording parameters, then run Capture.")

Initializing EoSens Creation 2.0 CX12 via EGrabber...
Successfully initialized EoSens camera.
Selected AOI:  (x=894, y=504, width=106, height=48)
Hardware AOI:  (x=880, y=504, width=128, height=48) [crop_x = 14]
Next: configure Recording parameters, then run Capture.


# Recording parameters
Set `camera_parameters = [[None, 0.020, 100]]`
Columns: `[FPS (None = query max at this AOI), exposure_ms, buffer_part_count]`
- `None` in the FPS slot automatically calculates and applies the maximum supported frame rate.
- `exposure_ms`: Exposure time in milliseconds (e.g., `0.020` ms = $20\,\mu\text{s}$).
- `buffer_part_count`: Number of frame parts bundled per GenTL buffer (default `100` prevents PCIe FIFO dropouts at high FPS).

In [4]:
# ---------------- RECORDING PARAMETERS: EDIT HERE ----------------
# Columns: [FPS (None for maximum), exposure_us, gain (1, 2, or 3), buffer_part_count]
camera_parameters = [[44100.0, 20.0, 3, 100]]
# -----------------------------------------------------------------

_recording_settings_valid = False
if not cams or N_cams != 1:
    raise RuntimeError("Connect camera and confirm an AOI before configuring recording.")

camera_parameters_requested = copy.deepcopy(camera_parameters)
recording_settings = []

for i, recording_cam in enumerate(cams):
    fps_req, exp_req_us, gain_req, parts_req = camera_parameters_requested[i]
    result = configure_recording_camera(
        recording_cam,
        frame_rate=fps_req,
        exposure_us=exp_req_us,
        gain=gain_req,
        buffer_part_count=parts_req
    )
    recording_settings.append(result)

    print(f"-------- CAMERA {i}: RECORDING PARAMETERS --------")
    print(f"User AOI (x, y, width, height): {tuple(result['aoi'])}")
    print(f"Padded Hardware AOI:            {tuple(result['hw_aoi'])} (crop_x={result['crop_x']})")
    print(f"Hardware maximum at this AOI:   {result['max_frame_rate_fps']:.2f} FPS")
    print(f"Configured frame rate:          {result['frame_rate_fps']:.2f} FPS")
    print(f"Exposure:                       {result['exposure_us']:.1f} µs ({result['exposure_ms']:.4f} ms)")
    print(f"Gain:                           {result['gain']}x")
    print(f"Buffer parts per buffer:        {result['buffer_part_count']}")

# Update downstream notebook globals
camera_parameters = [
    [s["frame_rate_fps"] if row[0] is None else float(row[0]),
     float(row[1]), int(row[2]), int(row[3])]
    for row, s in zip(camera_parameters_requested, recording_settings)
]

IMG_H = np.array([s["aoi"][3] for s in recording_settings], dtype=int)
IMG_W = np.array([s["aoi"][2] for s in recording_settings], dtype=int)
HW_W = np.array([s["hw_aoi"][2] for s in recording_settings], dtype=int)
CROP_X = np.array([s["crop_x"] for s in recording_settings], dtype=int)

frame_rate_eff = np.array([s["frame_rate_fps"] for s in recording_settings])
exposure_us_eff = np.array([s["exposure_us"] for s in recording_settings])
exposure_eff = np.array([s["exposure_ms"] for s in recording_settings])

capture_params = dict(
    camera_parameters=copy.deepcopy(camera_parameters),
    camera_parameters_requested=copy.deepcopy(camera_parameters_requested),
    IMG_H=IMG_H.copy(),
    IMG_W=IMG_W.copy(),
    HW_W=HW_W.copy(),
    CROP_X=CROP_X.copy(),
    frame_rate_eff=frame_rate_eff.copy(),
    exposure_eff=exposure_eff.copy(),
    exposure_us_eff=exposure_us_eff.copy(),
    buffer_part_count=recording_settings[0]["buffer_part_count"]
)
run_opt["capture_params"] = capture_params
run_opt["cam_params"]["camera_FPS"] = float(frame_rate_eff[0])
run_opt["cam_params"]["exposure"] = float(exposure_us_eff[0])

_recording_settings_valid = True
print(f"\nFinal configuration confirmed: camera_parameters = {camera_parameters}")

-------- CAMERA 0: RECORDING PARAMETERS --------
User AOI (x, y, width, height): (894, 504, 106, 48)
Padded Hardware AOI:            (880, 504, 128, 48) (crop_x=14)
Hardware maximum at this AOI:   44951.00 FPS
Configured frame rate:          44100.00 FPS
Exposure:                       20.0 µs (0.0200 ms)
Gain:                           3x
Buffer parts per buffer:        100

Final configuration confirmed: camera_parameters = [[44100.0, 20.0, 3, 100]]


# Capture
- `capture_sec`: Desired recording duration in seconds.
- `play_and_capture`: When `True`, synchronizes recording with audio playback from `wav_filename`.
- Frame-count based acquisition: Requests $N = \text{round}(\text{FPS} \times \text{capture\_sec})$ frames.
- Automatic contiguous memory unpacking ensures zero dropped frames.

In [18]:
# ---------------- CAPTURE PARAMETERS: EDIT HERE ----------------
capture_sec = 6.0          # Fallback duration if play_and_capture=False or file is missing
play_and_capture = True
# wav_filename = "audio_samples/golden_22K_9sec_sequence_22_repeats_32bit.wav"
wav_filename = "audio_samples/chirp_logarithmic_50_to_22000_22_repeats_32bit.wav"
AUDIO_PADDING_SEC = 1.5    # Extra recording time after audio ends
MAX_RECORDING_GIB = 80.0   # Upper RAM ceiling tailored for 96 GB systems
# ---------------------------------------------------------------

if not _recording_settings_valid:
    raise RuntimeError("Recording parameters must be configured before running capture.")

def get_audio_duration_sec(filepath: str) -> float:
    """Read duration for 16-bit, 24-bit, and 32-bit float WAVs without loading audio data into RAM."""
    # Method 1: scipy.io.wavfile with memory mapping (handles 32-bit IEEE float format 3 natively)
    try:
        sr, data = wavfile.read(filepath, mmap=True)
        return float(len(data)) / float(sr)
    except Exception:
        pass

    # Method 2: Standard wave module fallback for uncompressed integer PCM
    try:
        import wave
        with wave.open(filepath, 'rb') as wf:
            return float(wf.getnframes()) / float(wf.getframerate())
    except Exception:
        pass

    raise RuntimeError("Could not decode audio duration from WAV header.")

# 1. Resolve duration from the WAV audio file
recording_duration = float(capture_sec)
if play_and_capture and os.path.exists(wav_filename):
    try:
        wav_len = get_audio_duration_sec(wav_filename)
        recording_duration = wav_len + float(AUDIO_PADDING_SEC)
        print(f"Detected WAV: {wav_len:.2f} s (+{AUDIO_PADDING_SEC:.1f} s padding)")
        print(f"Target Time:  {recording_duration:.2f} s total duration")
    except Exception as exc:
        print(f"Warning: {exc}. Using manual fallback: {capture_sec:.2f} s")
else:
    print(f"Target Time:  {recording_duration:.2f} s (Manual duration)")

active_cam = cams[0]
fps_target = float(frame_rate_eff[0])
h_roi, w_roi = int(IMG_H[0]), int(IMG_W[0])
w_hw = int(capture_params["HW_W"][0])
crop_x = int(capture_params["CROP_X"][0])
parts_per_buf = int(capture_params["buffer_part_count"])

# Calculate exact frame quota and buffer allocation
total_frames = int(round(fps_target * recording_duration))
n_buffers = int(math.ceil(total_frames / parts_per_buf))

# Memory calculation: allocates ONLY the final user dimensions (no memory duplication)
user_gib = (total_frames * h_roi * w_roi) / (1024 ** 3)
hw_stream_gib = (n_buffers * parts_per_buf * h_roi * w_hw) / (1024 ** 3)

print(f"Target:       {total_frames:,} frames (Selected: {w_roi}x{h_roi}, Stream: {w_hw}x{h_roi}) @ {fps_target:.2f} FPS")
print(f"RAM Usage:    ~{user_gib:.2f} GiB allocated directly (Peak-safe on-the-fly cropping)")

if user_gib > MAX_RECORDING_GIB:
    raise MemoryError(
        f"Requested {user_gib:.2f} GiB exceeds safety ceiling ({MAX_RECORDING_GIB:.1f} GiB).\n"
        f"Reduce ROI height or sequence duration."
    )

class EoSensCaptureThread(Thread):
    def __init__(self, camera: EoSensCamera, n_bufs: int, parts: int, h: int, w_hw: int, crop_x: int, w_user: int, total_frames: int):
        super().__init__()
        self.camera = camera
        self.grabber = camera.grabber
        self.n_bufs = n_bufs
        self.parts = parts
        self.h = h
        self.w_hw = w_hw
        self.crop_x = crop_x
        self.w_user = w_user
        self.total_frames = total_frames
        
        # Preallocate ONLY the exact user-selected size to bypass double-allocation spikes
        self.final_frames = np.empty((total_frames, self.h, self.w_user), dtype=np.uint8)
        self.delivered_frames = 0
        self.exception = None

    def run(self):
        try:
            self.camera.grabber.flush_buffers()
            self.camera.grabber.start()
            frame_idx = 0
            for b_idx in range(self.n_bufs):
                with Buffer(self.grabber, timeout=5000) as buffer:
                    ptr = buffer.get_info(BUFFER_INFO_BASE, INFO_DATATYPE_PTR)
                    part_size = buffer.get_info(BUFFER_INFO_CUSTOM_PART_SIZE, INFO_DATATYPE_SIZET)
                    delivered = buffer.get_info(BUFFER_INFO_CUSTOM_NUM_DELIVERED_PARTS, INFO_DATATYPE_SIZET)

                    total_bytes = delivered * part_size
                    c_buf = ct.cast(ptr, ct.POINTER(ct.c_ubyte * total_bytes)).contents
                    np_buf = np.frombuffer(c_buf, count=total_bytes, dtype=np.uint8).reshape((delivered, self.h, self.w_hw))

                    remaining = self.total_frames - frame_idx
                    if remaining <= 0:
                        break
                    take = min(delivered, remaining)

                    # Crop hardware padding directly into the destination array
                    if self.w_user < self.w_hw:
                        self.final_frames[frame_idx : frame_idx + take] = np_buf[:take, :, self.crop_x : self.crop_x + self.w_user]
                    else:
                        self.final_frames[frame_idx : frame_idx + take] = np_buf[:take]

                    frame_idx += take
                    self.delivered_frames += delivered
        except Exception as exc:
            self.exception = exc
        finally:
            try:
                self.camera.grabber.stop()
            except Exception:
                pass

for count in [3, 2, 1]:
    print(f"Starting in {count}...")
    time.sleep(1.0)

imager = EoSensCaptureThread(active_cam, n_buffers, parts_per_buf, h_roi, w_hw, crop_x, w_roi, total_frames)

if play_and_capture and os.path.exists(wav_filename):
    winsound.PlaySound(wav_filename, winsound.SND_FILENAME | winsound.SND_ASYNC)

t_start = time.perf_counter()
imager.start()
imager.join()
t_end = time.perf_counter()

if play_and_capture:
    winsound.PlaySound(None, winsound.SND_PURGE)

if imager.exception is not None:
    raise RuntimeError(f"Capture encountered an error: {imager.exception}")

frame_recording = imager.final_frames

# Release internal camera/grabber references immediately to prevent resource lock
imager.camera = None
imager.grabber = None

wall_time = t_end - t_start
sustained_fps = len(frame_recording) / wall_time
throughput_mb = (frame_recording.nbytes / (1024 ** 2)) / wall_time

# Downstream globals and metadata
frames = frame_recording
capture_complete = True
capture_metadata = {
    'camera_settings': {
        'frame_rate_fps': fps_target,
        'exposure_us': float(exposure_us_eff[0]),
        'aoi': (AOI_X, AOI_Y, w_roi, h_roi),
        'hw_aoi': (int(capture_params["HW_W"][0]), h_roi)
    }
}

print(f"\n--- CAPTURE COMPLETED ---")
print(f"Acquired:       {len(frame_recording):,} / {total_frames:,} requested frames")
print(f"Array shape:    {frame_recording.shape} (H={h_roi}, W={w_roi})")
print(f"Wall-clock:     {wall_time:.4f} s (Requested duration: {recording_duration:.2f} s)")
print(f"Sustained FPS:  {sustained_fps:.2f} frames/sec")
print(f"Throughput:     {throughput_mb:.2f} MB/s")

Detected WAV: 226.20 s (+1.5 s padding)
Target Time:  227.70 s total duration
Target:       10,041,567 frames (Selected: 106x48, Stream: 128x48) @ 44100.00 FPS
RAM Usage:    ~47.58 GiB allocated directly (Peak-safe on-the-fly cropping)
Starting in 3...
Starting in 2...
Starting in 1...

--- CAPTURE COMPLETED ---
Acquired:       10,041,567 / 10,041,567 requested frames
Array shape:    (10041567, 48, 106) (H=48, W=106)
Wall-clock:     227.9122 s (Requested duration: 227.70 s)
Sustained FPS:  44058.93 frames/sec
Throughput:     213.79 MB/s


# Video verification
Inspect the acquired recording using a non-blocking interactive OpenCV video player, or export an uncompressed AVI video.

In [19]:
def play_captured_video(frames: np.ndarray, playback_fps: float = 30.0, scale="auto"):
    """Interactive video player with top text banner, auto-scaling, and custom +/- and */ scrubbing."""
    n_frames = len(frames)
    if n_frames == 0:
        print("No frames to display.")
        return

    h_raw, w_raw = frames.shape[1], frames.shape[2]

    # Calculate optimal pixel scaling so small AOIs are large enough to see clearly
    if scale == "auto" or scale is None:
        # Scale height to at least ~250 px, but keep window width within 1280 px
        scale_by_h = max(1, 260 // h_raw)
        scale_by_w = max(1, 1280 // w_raw)
        scale_factor = max(1, min(scale_by_h, scale_by_w))
        if h_raw <= 32 and scale_factor < 8:
            scale_factor = max(scale_factor, 8)
    else:
        scale_factor = max(1, int(scale))

    scaled_w = w_raw * scale_factor
    scaled_h = h_raw * scale_factor

    # Top header bar height where status text is rendered
    header_h = 55
    canvas_w = max(scaled_w, 640)
    canvas_h = scaled_h + header_h
    x_offset = (canvas_w - scaled_w) // 2

    win_name = "Captured Video [Space: Pause | +/-: 1 frame | */: 10 frames | Esc: Exit]"
    cv2.namedWindow(win_name, cv2.WINDOW_AUTOSIZE)

    # Bring window to front
    try:
        import ctypes
        hwnd = ctypes.windll.user32.FindWindowW(None, win_name)
        if hwnd:
            ctypes.windll.user32.SetWindowPos(hwnd, -1, 0, 0, 0, 0, 0x0002 | 0x0001)
    except Exception:
        pass

    idx = 0
    paused = True  # Start paused to allow immediate manual scrubbing
    delay = max(1, int(1000.0 / playback_fps))

    print(f"\nDisplaying {w_raw}x{h_raw} AOI scaled {scale_factor}x -> {scaled_w}x{scaled_h} px")
    print("Controls:")
    print("  [Space]       : Pause / Resume")
    print("  [+] or [=]    : Step  +10 frames")
    print("  [-]           : Step  -10 frames")
    print("  [*]           : Jump +100 frames")
    print("  [/]           : Jump -100 frames")
    print("  [Right] / [d] : Step   +1 frame")
    print("  [Left]  / [a] : Step   -1 frame")
    print("  [Esc] / [q]   : Exit viewer\n")

    while True:
        # 1. Prepare raw frame and stretch contrast
        raw_frame = stretch_contrast(frames[idx].copy())
        frame_bgr = cv2.cvtColor(raw_frame, cv2.COLOR_GRAY2BGR)

        # 2. Resize with nearest neighbor to keep pixels sharp
        scaled_frame = cv2.resize(frame_bgr, (scaled_w, scaled_h), interpolation=cv2.INTER_NEAREST)

        # 3. Build canvas with black header panel above the frame
        canvas = np.zeros((canvas_h, canvas_w, 3), dtype=np.uint8)
        canvas[header_h:header_h + scaled_h, x_offset:x_offset + scaled_w] = scaled_frame

        # Optional: subtle separator line between header and frame
        cv2.line(canvas, (0, header_h - 1), (canvas_w, header_h - 1), (60, 60, 60), 1)

        # 4. Render status text in the header area ABOVE the frame
        status_text = f"Frame: {idx + 1} / {n_frames}   {'[PAUSED]' if paused else '[PLAYING]'}"
        cv2.putText(canvas, status_text, (18, 36), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2, cv2.LINE_AA)

        cv2.imshow(win_name, canvas)

        key = cv2.waitKeyEx(0 if paused else delay)
        k_ascii = key & 0xFF

        # Exit conditions
        if k_ascii in (27, 13, ord('q'), ord('Q')):  # Esc, Enter, or 'q'
            break

        # Play / Pause toggle
        elif k_ascii == 32:
            paused = not paused

        # Step 1 frame (+ and - or Left/Right arrows / a/d)
        elif k_ascii in (43, ord('='), ord('+')):  # '+' or '='
            idx = (idx + 10) % n_frames
        elif k_ascii in (45, ord('-')):            # '-'
            idx = (idx - 10) % n_frames
        elif key in (2555904, 65363) or k_ascii in (83, ord('d'), ord('D')):  # Right arrow / 'd'
            idx = (idx + 1) % n_frames
        elif key in (2424832, 65361) or k_ascii in (81, ord('a'), ord('A')):  # Left arrow / 'a'
            idx = (idx - 1) % n_frames

        # Jump 10 frames (* and /)
        elif k_ascii == 42 or key == 42:           # '*'
            idx = (idx + 100) % n_frames
        elif k_ascii == 47 or key == 47:           # '/'
            idx = (idx - 100) % n_frames

        # Continuous playback
        elif not paused:
            idx = (idx + 1) % n_frames

    cv2.destroyWindow(win_name)

# Launch interactive player with automatic optimal scaling
play_captured_video(frame_recording, playback_fps=30.0, scale="auto")


Displaying 106x48 AOI scaled 5x -> 530x240 px
Controls:
  [Space]       : Pause / Resume
  [+] or [=]    : Step  +10 frames
  [-]           : Step  -10 frames
  [*]           : Jump +100 frames
  [/]           : Jump -100 frames
  [Right] / [d] : Step   +1 frame
  [Left]  / [a] : Step   -1 frame
  [Esc] / [q]   : Exit viewer



# Reconstruct Audio from Interferometry

In [ ]:
# ----------------- NAMING CONFIGURATION -----------------
NAME_PREFIX = "drum_interferometry"   # Prefix prepended to all saved files and folders
# --------------------------------------------------------

if 'frames' not in globals() and 'frame_recording' in globals():
    frames = frame_recording
    capture_complete = True

if not globals().get('capture_complete', False) or 'frames' not in globals():
    raise RuntimeError("No completed capture found in memory. Run the Capture cell first.")

# Extract base name from wav_filename
wav_base_name = None
if 'wav_filename' in globals() and wav_filename:
    try:
        p = Path(wav_filename)
        stem = p.stem.strip()
        if stem:
            wav_base_name = stem
    except Exception:
        pass

raw_stem = wav_base_name if wav_base_name else "test"
prefix_str = f"{NAME_PREFIX.strip()}_" if NAME_PREFIX and NAME_PREFIX.strip() else ""
base_name = f"{prefix_str}{raw_stem}"

exp_timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
experiment_name = f"{base_name}_{exp_timestamp}"

output_dir = Path("./recover_alg/SOUND_RECOVERIES") / experiment_name
output_dir.mkdir(parents=True, exist_ok=True)

def get_safe_path(p: Path) -> str:
    resolved_str = str(p.resolve())
    if os.name == 'nt' and len(resolved_str) >= 240 and not resolved_str.startswith('\\\\?\\'):
        return '\\\\?\\' + resolved_str
    return resolved_str

if 'capture_metadata' in globals() and 'camera_settings' in capture_metadata:
    fps = float(capture_metadata['camera_settings']['frame_rate_fps'])
elif 'frame_rate_eff' in globals():
    fps = float(frame_rate_eff[0])
else:
    fps = float(run_opt['cam_params']['camera_FPS'])

num_frames, H, W = frames.shape
n_pixels = H * W
frame_indices = np.arange(num_frames)

print(f"Base name: {experiment_name}")
print(f"Processing: {W}x{H} across {num_frames:,} frames ({frames.nbytes/(1024**3):.2f} GiB) @ {fps:.2f} FPS...")

# ---------------- STEP 1: COMPUTE SPATIAL MODES ON SUBSET ----------------
sample_step = max(1, num_frames // 100000)
sample_indices = np.arange(0, num_frames, sample_step)
print(f"Fitting SVD spatial modes on {len(sample_indices):,} representative frames...")

X_sample = frames[sample_indices].reshape(len(sample_indices), -1).astype(np.float32)
mean_pixel = np.mean(X_sample, axis=0, dtype=np.float32)
X_sample -= mean_pixel

svd = TruncatedSVD(n_components=6, random_state=42)
svd.fit(X_sample)
spatial_modes = svd.components_  # Shape: (6, H * W)
del X_sample
gc.collect()

# ---------------- STEP 2: STREAM BATCH PROJECTION OVER TIME ----------------
print("Projecting temporal components in batches...")
primary_motion = np.empty(num_frames, dtype=np.float32)
secondary_motion = np.empty(num_frames, dtype=np.float32)

comp1 = spatial_modes[0]
comp2 = spatial_modes[1]

batch_size = 100000
for start in range(0, num_frames, batch_size):
    end = min(start + batch_size, num_frames)
    batch_raw = frames[start:end].reshape(end - start, -1).astype(np.float32)
    batch_centered = batch_raw - mean_pixel
    
    primary_motion[start:end] = np.dot(batch_centered, comp1)
    secondary_motion[start:end] = np.dot(batch_centered, comp2)

del batch_raw, batch_centered
gc.collect()

primary_motion -= np.mean(primary_motion)
secondary_motion -= np.mean(secondary_motion)

# ---------------- STEP 3: PHASE UNWRAPPING & AUDIO FILTER ----------------
print("Demodulating phase...")
raw_signal = np.unwrap(np.arctan2(secondary_motion, primary_motion))

low_f = 10.0
nyquist = fps / 2.0
cutoff = min(low_f, nyquist * 0.5)
b, a = butter(4, cutoff, btype="highpass", fs=fps)
filtered_signal = filtfilt(b, a, raw_signal)

audio_signal = filtered_signal.astype(np.float32)
peak = np.max(np.abs(audio_signal))
if peak > 0:
    audio_signal /= peak

# Save files with custom prefix and timestamp
wav_path = output_dir / f"{base_name}_recovered_{exp_timestamp}.wav"
npy_path = output_dir / f"{base_name}_recovered_{exp_timestamp}.npy"
wavfile.write(get_safe_path(wav_path), int(round(fps)), audio_signal)
np.save(get_safe_path(npy_path), audio_signal)

print(f"Saved audio: {wav_path}")
print(f"Saved array: {npy_path}")

# ---------------- AUDIO PLAYER ----------------
playback_fs = 44100
num_playback_samples = int(round(len(audio_signal) * playback_fs / fps))
audio_playback = resample(audio_signal, num_playback_samples).astype(np.float32)
playback_peak = np.max(np.abs(audio_playback))
if playback_peak > 0:
    audio_playback /= playback_peak

print("\n--- RECOVERED AUDIO PREVIEW ---")
display(Audio(audio_playback, rate=playback_fs))

# ---------------- PLOTS ----------------
fig, axes = plt.subplots(3, 2, figsize=(10, 4.0), constrained_layout=True)
fig.suptitle(f"Spatial SVD Components — {base_name}", fontsize=12, fontweight="bold")
for comp_idx in range(6):
    r, c = divmod(comp_idx, 2)
    ax = axes[r, c]
    comp_img = spatial_modes[comp_idx, :].reshape(H, W)
    local_vmax = np.max(np.abs(comp_img))
    ax.imshow(comp_img, cmap="seismic", vmin=-local_vmax, vmax=local_vmax)
    ax.set_title(f"Component {comp_idx + 1}", fontsize=10, fontweight="bold", pad=3)
    ax.axis("off")
plt.savefig(get_safe_path(output_dir / f"{base_name}_spatial_grid_{exp_timestamp}.png"), dpi=300, bbox_inches="tight")
plt.show()

scatter_stride = max(1, num_frames // 50000)
fig_scatter, ax_scatter = plt.subplots(figsize=(5.5, 4.5))
scatter = ax_scatter.scatter(
    primary_motion[::scatter_stride], secondary_motion[::scatter_stride],
    c=frame_indices[::scatter_stride], cmap="viridis",
    s=2.0, alpha=0.75, edgecolors="none"
)
plt.colorbar(scatter, ax=ax_scatter, label="Frame Index")
ax_scatter.set_title(f"SVD Motion Trajectory — {base_name}", fontsize=11, fontweight="bold", pad=8)
ax_scatter.set_xlabel("Component 1", fontsize=10)
ax_scatter.set_ylabel("Component 2", fontsize=10)
ax_scatter.set_aspect("equal", adjustable="datalim")
ax_scatter.grid(True, linestyle="--", alpha=0.3)
plt.tight_layout()
plt.savefig(get_safe_path(output_dir / f"{base_name}_svd_scatter_{exp_timestamp}.png"), dpi=300, bbox_inches="tight")
plt.show()

n = len(audio_signal)
freqs = np.fft.rfftfreq(n, d=1.0 / fps)
fft_mag = (2.0 / n) * np.abs(np.fft.rfft(audio_signal))

fig, (ax_zoom, ax_full) = plt.subplots(1, 2, figsize=(14, 3.8), constrained_layout=True)
audio_limit = min(5000.0, nyquist)
zoom_mask = freqs <= audio_limit
ax_zoom.plot(freqs[zoom_mask], fft_mag[zoom_mask], color="#1f77b4", lw=1.2)
ax_zoom.set_title(f"Acoustic Band (0 – {int(audio_limit):,} Hz)", fontsize=11, fontweight="bold")
ax_zoom.set_xlabel("Frequency (Hz)", fontsize=10)
ax_zoom.set_ylabel("Magnitude", fontsize=10)
ax_zoom.set_xlim(0, audio_limit)
ax_zoom.grid(True, linestyle="--", alpha=0.5)

ax_full.plot(freqs, fft_mag, color="#7f7f7f", lw=0.9)
ax_full.set_title(f"Full Spectrum (0 – {int(nyquist):,} Hz)", fontsize=11, fontweight="bold")
ax_full.set_xlabel("Frequency", fontsize=10)
ax_full.set_xlim(0, nyquist)
ax_full.grid(True, linestyle="--", alpha=0.4)
plt.savefig(get_safe_path(output_dir / f"{base_name}_fft_spectrum_{exp_timestamp}.png"), dpi=300, bbox_inches="tight")
plt.show()

Base name: interferometry_chirp_logarithmic_50_to_22000_22_repeats_32bit_20260924_002505
Processing: 106x48 across 10,041,567 frames (47.58 GiB) @ 44100.00 FPS...
Fitting SVD spatial modes on 100,416 representative frames...
Projecting temporal components in batches...


# Save recording and metadata
Persists raw numpy frames and complete experiment metadata.

In [21]:
## 7. Save recording and metadata

# ----------------- NAMING CONFIGURATION -----------------
NAME_PREFIX = "drum_interferometry"   # Prefix prepended to all saved files and folders
# --------------------------------------------------------

# Derive base name from wav_filename
wav_base_name = None
if 'wav_filename' in globals() and wav_filename:
    try:
        p = Path(wav_filename)
        stem = p.stem.strip()
        if stem:
            wav_base_name = stem
    except Exception:
        pass

raw_stem = wav_base_name if wav_base_name else "recording"
prefix_str = f"{NAME_PREFIX.strip()}_" if NAME_PREFIX and NAME_PREFIX.strip() else ""
base_name = f"{prefix_str}{raw_stem}"

exp_timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

output_dir = Path("recover_alg/real_data/frame_recordings") / f"{base_name}_{exp_timestamp}"
output_dir.mkdir(parents=True, exist_ok=True)

raw_filename = f"{base_name}_{exp_timestamp}.npy"
raw_path = output_dir / raw_filename
meta_path = output_dir / f"metadata_{base_name}_{exp_timestamp}.npz"

def get_safe_path(p: Path) -> str:
    resolved_str = str(p.resolve())
    if os.name == 'nt' and len(resolved_str) >= 240 and not resolved_str.startswith('\\\\?\\'):
        return '\\\\?\\' + resolved_str
    return resolved_str

t_save_start = time.time()
print(f"Base name with prefix: {base_name}")
print(f"Saving {len(frame_recording):,} frames ({frame_recording.nbytes/(1024**3):.2f} GiB) to {raw_path.resolve()}...")

np.save(get_safe_path(raw_path), frame_recording)
np.savez(
    get_safe_path(meta_path),
    experiment_name=base_name,
    timestamp=exp_timestamp,
    wav_filename=globals().get('wav_filename', ''),
    run_opt=run_opt,
    capture_params=capture_params,
    accepted_aoi=accepted_aoi
)

t_save_elapsed = time.time() - t_save_start
print(f"Saved successfully in {t_save_elapsed:.2f} seconds.")

Base name: chirp_logarithmic_50_to_22000_22_repeats_32bit
Saving 10041567 frames to C:\Users\matan\Weizmann Institute Dropbox\Matan Kichler\PROJECT_Interferometry[Matan]\recover_alg\real_data\frame_recordings\chirp_logarithmic_50_to_22000_22_repeats_32bit_20260924_002213\chirp_logarithmic_50_to_22000_22_repeats_32bit_20260924_002213.npy...


FileNotFoundError: [Errno 2] No such file or directory: 'recover_alg\\real_data\\frame_recordings\\chirp_logarithmic_50_to_22000_22_repeats_32bit_20260924_002213\\metadata_chirp_logarithmic_50_to_22000_22_repeats_32bit_20260924_002213.npz'